In [ ]:
  # Downloaded the Data from Github


In [ ]:
pip install gitpython

In [ ]:
from git import repo

In [ ]:
!git clone https://github.com/PhonePe/pulse.git

Cloning into 'pulse'...
remote: Enumerating objects: 17904, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 17904 (delta 19), reused 17 (delta 17), pack-reused 17855 (from 2)
Receiving objects: 100% (17904/17904), 26.13 MiB | 6.61 MiB/s, done.
Resolving deltas: 100% (8723/8723), done.
Updating files: 100% (9029/9029), done.


In [ ]:
# Aggregated Transaction Data
import os
import json
import pandas as pd

agg_tr_path = "/content/pulse/data/aggregated/transaction/country/india/state"

agg_trans = []

for state in os.listdir(agg_tr_path):
  state_path = os.path.join(agg_tr_path, state)
  for year in os.listdir(state_path):
    year_path = os.path.join(state_path, year)
    for file in os.listdir(year_path):
      if file.endswith(".json"):
        with open(os.path.join(year_path, file)) as f:
          data = json.load(f)
          if data['data']['transactionData']:
            for record in data['data']['transactionData']:
              transaction_type = record['name']
              transaction_count = record['paymentInstruments'][0]['count']
              transaction_amount = record['paymentInstruments'][0]['amount']
              agg_trans.append([state, year, file.split('.')[0], transaction_type, transaction_count, transaction_amount])


df_agg_trans = pd.DataFrame(agg_trans, columns=["state", "year", "quarter", "transaction_type", "transaction_count", "transaction_amount"])

df_agg_trans.to_csv("aggregated_transaction.csv", index=False)

print("aggregated_transaction:", len(df_agg_trans), "rows")

aggregated_transaction: 5034 rows


In [ ]:
# Aggregated User Data

import os
import json
import pandas as pd

# Path to aggregated user JSON files
agg_user_path = "/content/pulse/data/aggregated/user/country/india/state"

agg_user = []

for state in os.listdir(agg_user_path):
    state_path = os.path.join(agg_user_path, state)
    if not os.path.isdir(state_path):
        continue
    for year in os.listdir(state_path):
        year_path = os.path.join(state_path, year)
        if not os.path.isdir(year_path):
            continue
        for file in os.listdir(year_path):
            if file.endswith(".json"):
                file_path = os.path.join(year_path, file)
                with open(file_path, "r") as f:
                    data = json.load(f)
                quarter = file.split('.')[0]
                if data.get("data"):
                    # Safe extraction of aggregated user metrics
                    reg_users = data["data"].get("registeredUsers", 0)
                    app_opens = data["data"].get("appOpens", 0)
                    users_by_device = data["data"].get("usersByDevice", [])

                    if users_by_device:
                        for d in users_by_device:
                            brand = d.get("brand", None)
                            count = d.get("count", 0)
                            agg_user.append([state, int(year), int(quarter), brand, count, reg_users, app_opens])
                    else:
                        # If no device info, fill with None and 0
                        agg_user.append([state, int(year), int(quarter), None, 0, reg_users, app_opens])

# Create DataFrame
df_agg_user = pd.DataFrame(
    agg_user,
    columns=["state", "year", "quarter", "device_brand", "user_count", "registered_users", "app_opens"]
)

# Save CSV
df_agg_user.to_csv("aggregated_user.csv", index=False)

print("Aggregated User CSV saved:", len(df_agg_user), "rows")


Aggregated User CSV saved: 7128 rows


In [ ]:
df_agg_user.head(10)

,state,year,quarter,device_brand,user_count,registered_users,app_opens
0,puducherry,2024,2,None,0,0,0
1,puducherry,2024,4,None,0,0,0
2,puducherry,2024,1,None,0,0,0
3,puducherry,2024,3,None,0,0,0
4,puducherry,2019,2,Xiaomi,36841,0,0
5,puducherry,2019,2,Samsung,30450,0,0
6,puducherry,2019,2,Vivo,24993,0,0
7,puducherry,2019,2,Oppo,14342,0,0
8,puducherry,2019,2,Huawei,6414,0,0
9,puducherry,2019,2,Apple,6391,0,0


In [ ]:
# Aggregated Insurance Data

agg_ins_path = "/content/pulse/data/aggregated/insurance/country/india/state"
agg_ins = []

for state in os.listdir(agg_ins_path):
    state_path = os.path.join(agg_ins_path, state)
    for year in os.listdir(state_path):
        year_path = os.path.join(state_path, year)
        for file in os.listdir(year_path):
            if file.endswith(".json"):
                quarter = file.replace(".json", "")
                file_path = os.path.join(year_path, file)
                with open(file_path, "r") as f:
                    data = json.load(f)

                if not data.get("data"):
                    continue

                for key in ["transactionData", "insuranceData", "aggregated"]:
                    if data["data"].get(key):
                        for rec in data["data"][key]:
                            name = rec.get("name", "Insurance")
                            if "paymentInstruments" in rec and len(rec["paymentInstruments"]) > 0:
                                count = rec["paymentInstruments"][0].get("count", 0)
                                amount = rec["paymentInstruments"][0].get("amou   nt", 0)
                                agg_ins.append([state, year, quarter, name, count, amount])
                        break  # stop after first valid key

df_agg_ins = pd.DataFrame(
    agg_ins,
    columns=["state", "year", "quarter", "insurance_type", "policy_count", "policy_amount"]
)

df_agg_ins.to_csv("aggregated_insurance.csv", index=False)
print(" Aggregated insurance:", len(df_agg_ins), "rows")


 Aggregated insurance: 682 rows


In [ ]:
# Map Transaction Data

map_tr_path = "/content/pulse/data/map/transaction/hover/country/india/state/"

map_tr = []

for state in os.listdir(map_tr_path):
  state_path = os.path.join(map_tr_path, state)
  for year in os.listdir(state_path):
    year_path = os.path.join(state_path, year)
    for file in os.listdir(year_path):
      if file.endswith(".json"):
        quarter = file.strip(".json")
        with open(os.path.join(year_path, file), "r") as f:
          data = json.load(f)
          if data["data"].get("hoverDataList"):
            for district in data["data"]["hoverDataList"]:
              dname = district["name"]
              count = district["metric"][0]["count"]
              amount = district["metric"][0]["amount"]
              map_tr.append([state, dname, int(year), int(quarter), count, amount])

          else:
            print(f" No data in file: {file_path}")

df_map_trans = pd.DataFrame(map_tr, columns=["state", "district", "year", "quarter", "transaction_count", "transaction_amount"])
df_map_trans.to_csv("map_transaction.csv", index=False)

print("Map transaction:", len(df_map_trans), "rows")

Map transaction: 20604 rows


In [ ]:
# Map User Data

map_user_path = "/content/pulse/data/map/user/hover/country/india/state"

map_user = []

for state in os.listdir(map_user_path):
  state_path = os.path.join(map_user_path, state)
  for year in os.listdir(state_path):
    year_path = os.path.join(state_path, year)
    for file in os.listdir(year_path):
      if file.endswith(".json"):
        with open(os.path.join(year_path, file), "r") as f:
          data = json.load(f)
          quarter = file.split('.')[0]
          if data.get("data"):
            reg_users = data["data"].get("registeredUsers", 0)
            app_opens = data["data"].get("appOpens", 0)
            users_by_device = data["data"].get("usersByDevice")
            if users_by_device:
              for d in users_by_device:
                brand = d["brand"]
                count = d["count"]
                map_user.append([state,int(year),int(quarter),brand,count,reg_users,app_opens])
            else:

              map_user.append([state, int(year), int(quarter), None, 0, reg_users, app_opens])


df_map_user = pd.DataFrame(map_user, columns=["state", "year", "quarter", "device_brand", "user_count", "registered_users", "app_opens"])

df_map_user.to_csv("map_user.csv", index=False)

print("Map User:", len(df_map_user), "rows")

Map User: 1008 rows


In [ ]:
# Map Insurance Data

map_ins_path = "/content/pulse/data/map/insurance/country/india/state"
map_ins = []

for state in os.listdir(map_ins_path):
    state_path = os.path.join(map_ins_path, state)
    for year in os.listdir(state_path):
        year_path = os.path.join(state_path, year)
        for file in os.listdir(year_path):
            if file.endswith(".json"):
                quarter = file.replace(".json", "")
                file_path = os.path.join(year_path, file)
                with open(file_path, "r") as f:
                    data = json.load(f)

                if not data.get("data"):
                    continue

                for key in ["transactionData", "insuranceData", "map"]:
                    if data["data"].get(key):
                        for rec in data["data"][key]:
                            name = rec.get("name", "Insurance")
                            if "paymentInstruments" in rec and len(rec["paymentInstruments"]) > 0:
                                count = rec["paymentInstruments"][0].get("count", 0)
                                amount = rec["paymentInstruments"][0].get("amount", 0)
                                map_ins.append([state, year, quarter, name, count, amount])
                        break  # stop after first valid key

    df_map_ins = pd.DataFrame(
    map_ins,
    columns=["state", "year", "quarter", "insurance_type", "policy_count", "policy_amount"]
)

df_map_ins.to_csv("map_insurance.csv", index=False)
print(" Map insurance:", len(df_map_ins), "rows")

 Map insurance: 0 rows


In [ ]:
# Confirming Map insurance: 0 rows

import glob, json

files = glob.glob("/content/pulse/data/map/insurance/country/india/state/*/*/*.json")
count_with_data = 0

for f in files:
  data = json.load(open(f))
  if data.get("data") and data["data"].get("hoverDataList"):
    count_with_data += 1

print("Files with actual map insurance data:", count_with_data)


Files with actual map insurance data: 0


In [ ]:
# Top Transaction Data

top_tr_path = "/content/pulse/data/top/transaction/country/india/state/"

top_tr = []

for state in os.listdir(top_tr_path):
    state_path = os.path.join(top_tr_path, state)
    for year in os.listdir(state_path):
        year_path = os.path.join(state_path, year)
        for file in os.listdir(year_path):
            if file.endswith(".json"):
                quarter = file.replace(".json", "")
                file_path = os.path.join(year_path, file)

                with open(file_path, "r") as f:
                    data = json.load(f)

                if not data.get("data"):
                    continue

                for key in ["states", "districts", "pincodes"]:
                    if data["data"].get(key):
                        for rec in data["data"][key]:
                            name = rec.get("entityName", "Unknown")
                            count = rec["metric"].get("count", 0)
                            amount = rec["metric"].get("amount", 0)
                            top_tr.append([state, int(year), int(quarter), key, name, count, amount])

df_top_trans = pd.DataFrame(
    top_tr,
    columns=["state", "year", "quarter", "level", "entity_name", "transaction_count", "transaction_amount"]
)

df_top_trans.to_csv("top_transaction.csv", index=False)
print(" Top Transaction:", len(df_top_trans), "rows")


 Top Transaction: 18295 rows


In [ ]:
# Top Insurance Data

top_ins_path = "/content/pulse/data/top/insurance/country/india/state"
top_ins = []

for state in os.listdir(top_ins_path):
    state_path = os.path.join(top_ins_path, state)
    for year in os.listdir(state_path):
        year_path = os.path.join(state_path, year)
        for file in os.listdir(year_path):
            if file.endswith(".json"):
                quarter = file.replace(".json", "")
                file_path = os.path.join(year_path, file)
                with open(file_path, "r") as f:
                    data = json.load(f)

                if not data.get("data"):
                    continue

                for key in ["transactionData", "insuranceData", "top"]:
                    if data["data"].get(key):
                        for rec in data["data"][key]:
                            name = rec.get("name", "Insurance")
                            if "paymentInstruments" in rec and len(rec["paymentInstruments"]) > 0:
                                count = rec["paymentInstruments"][0].get("count", 0)
                                amount = rec["paymentInstruments"][0].get("amount", 0)
                                top_ins.append([state, year, quarter, name, count, amount])
                        break  # stop after first valid key

df_top_ins = pd.DataFrame(
    top_ins,
    columns=["state", "year", "quarter", "insurance_type", "policy_count", "policy_amount"]
)

df_top_ins.to_csv("top_insurance.csv", index=False)
print(" Top insurance:", len(df_top_ins), "rows")


 Top insurance: 0 rows


In [ ]:
# Confirming Top insurance: 0 rows

import glob, json

files = glob.glob("/content/pulse/data/top/insurance/country/india/state/*/*/*.json")
count_with_data = 0

for f in files:
  data = json.load(open(f))
  if data.get("data") and data["data"].get("hoverDataList"):
    count_with_data += 1

print("Files with actual top insurance data:", count_with_data)

Files with actual top insurance data: 0


In [ ]:
# Top User Data

top_user_path = "/content/pulse/data/top/user/country/india/state"

top_user = []

for state in os.listdir(top_user_path):
  state_path = os.path.join(top_user_path, state)
  for year in os.listdir(state_path):
    year_path = os.path.join(state_path, year)
    for file in os.listdir(year_path):
      if file.endswith(".json"):
        with open(os.path.join(year_path, file), "r") as f:
          data = json.load(f)
          quarter = file.split('.')[0]
          if data.get("data"):
            reg_users = data["data"].get("registeredUsers", 0)
            app_opens = data["data"].get("appOpens", 0)
            users_by_device = data["data"].get("usersByDevice")
            if users_by_device:
              for d in users_by_device:
                brand = d["brand"]
                count = d["count"]
                top_user.append([state,int(year),int(quarter),brand,count,reg_users,app_opens])
            else:

              top_user.append([state, int(year), int(quarter), None, 0, reg_users, app_opens])


df_top_user = pd.DataFrame(top_user, columns=["state", "year", "quarter", "device_brand", "user_count", "registered_users", "app_opens"])

df_top_user.to_csv("top_user.csv", index=False)

print("Top User:", len(df_top_user), "rows")

Top User: 1008 rows


In [ ]:
# File Download

from google.colab import files

for f in ["aggregated_transaction.csv", "aggregated_insurance.csv", "aggregated_user.csv", "map_transaction.csv", "top_transaction.csv"]:
  files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import glob

for file in glob.glob("*.csv"):
  df = pd.read_csv(file).drop_duplicates().fillna(0)
  df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
  df.to_csv("cleaned_" + file, index=False)
  print(f"cleaned {file}")

cleaned aggregated_user.csv


In [ ]:
# File Download

from google.colab import files

for f in ["cleaned_aggregated_transaction.csv", "cleaned_aggregated_insurance.csv", "cleaned_aggregated_user.csv", "cleaned_map_transaction.csv", "cleaned_map_user.csv", "cleaned_map_insurance.csv", "cleaned_top_transaction.csv", "cleaned_top_insurance.csv", "cleaned_top_user.csv"]:
  files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd

df = pd.read_csv("cleaned_aggregated_insurance.csv")

df["policy_amount"] = (df["policy_amount"].astype(str).str.replace(",", "", regex=False).astype(float))

print(df.dtypes)
df.to_csv("cleaned_aggregated_insurance.csv", index=False)
print("commas removed and cleaned file saved")


state              object
year                int64
quarter             int64
insurance_type     object
policy_count        int64
policy_amount     float64
dtype: object
commas removed and cleaned file saved


In [ ]:
# File Download

from google.colab import files

for f in ["cleaned_aggregated_insurance.csv"]:
  files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd

df = pd.read_csv("cleaned_aggregated_transaction.csv")

df["transaction_amount"] = (df["transaction_amount"].astype(str).str.replace(",", "", regex=False).astype(float))

print(df.dtypes)
df.to_csv("cleaned_aggregated_transaction.csv", index=False)
print("commas removed and cleaned file saved")

state                  object
year                    int64
quarter                 int64
transaction_type       object
transaction_count       int64
transaction_amount    float64
dtype: object
commas removed and cleaned file saved


In [ ]:
# File Download

from google.colab import files

for f in ["cleaned_aggregated_transaction.csv"]:
  files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# File Download

from google.colab import files

for f in ["cleaned_aggregated_user.csv"]:
  files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>